# Super-Convergence

Часть 1: MNIST, CIFAR-10 и регуляризация. Часть 2: перенос расписания LR на DQN в LunarLander. Запускаем разделы по очереди; параметры фиксируем до просмотра test.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from train import load_data, train

torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
data = load_data(device)
{name: len(labels) for name, (_, labels) in data.items()}

## Обучение

Три сида. также хотим сравнивать при равном бюджете, сохраню результат бейзлайна на 12 эпохе

In [ ]:
runs = [("baseline", 85, None), ("onecycle", 12, 0.1), ("low_peak", 12, 0.02)]
seeds = [17, 42, 103]
results, histories = [], []

for index, seed in enumerate(seeds):
    for name, epochs, max_lr in runs[index:] + runs[:index]:
        history, scores = train(data, name, epochs, max_lr, seed, output="results/notebook")
        histories.append(history.assign(method=name, seed=seed))
        results.append(scores)

results = pd.concat(results, ignore_index=True)
histories = pd.concat(histories, ignore_index=True)

In [ ]:
results.groupby("method").agg(
    accuracy=("test_accuracy", "mean"),
    std=("test_accuracy", "std"),
    seconds=("train_seconds", "mean"),
).round(3)

In [ ]:
for name, group in histories.groupby("method"):
    curve = group.groupby("epoch").validation_accuracy.mean()
    plt.plot(curve.index, curve, label=name)

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy, %")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Вывод

Сопоставим точность, время обучения и разброс между seed. Отдельно посмотрим, что изменилось при уменьшении пикового learning rate.

## Loss и нормы

`train_loss` измеряется во время обучения, `train_eval_loss` — в режиме eval на фиксированных первых 5000 train-примерах без аугментации. Разрыв `validation_loss − train_eval_loss` сравниваем при одинаковом режиме оценки. Нормы считаются для весов сверток и линейных слоев, без bias и BatchNorm; RMS позволяет сопоставлять слои разного размера.

`grad_l2` — градиент data loss без weight decay; `update_ratio` учитывает фактическое изменение весов, включая momentum и weight decay.

In [ ]:
from pathlib import Path

def plot_curves(frame, x, columns):
    fig, axes = plt.subplots(1, len(columns), figsize=(5 * len(columns), 3.5), squeeze=False)
    for axis, column in zip(axes[0], columns):
        for name, group in frame.groupby("method"):
            curve = group.groupby(x)[column].agg(["mean", "std"])
            axis.plot(curve.index, curve["mean"], label=name)
            axis.fill_between(curve.index, curve["mean"] - curve["std"].fillna(0), curve["mean"] + curve["std"].fillna(0), alpha=0.15)
        axis.set(xlabel=x, ylabel=column)
        axis.grid(alpha=0.2)
    axes[0, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def read_diagnostics(folder):
    return pd.concat([pd.read_csv(p).assign(method=p.parent.parent.name, seed=int(p.parent.name))
                      for p in Path(folder).glob("*/*/diagnostics.csv")], ignore_index=True)

plot_curves(histories, "epoch", ["train_eval_loss", "validation_loss", "gap"])
plot_curves(read_diagnostics("results/notebook"), "step", ["weight_l2", "grad_l2", "update_ratio"])

## CIFAR-10

ResNet-20 с downsampling через average pooling и дополнением каналов нулями. 45 000 train / 5 000 validation / 10 000 test; случайные crop и flip только при обучении. Сравним cosine и 1cycle за 50 эпох, затем короткий 1cycle за 20 эпох. У обоих 1cycle одинаковые начальный LR=0,02 и пик=0,2. Weight decay=0,0005, batch size=128.

In [ ]:
cifar = load_data(device, dataset="cifar10")
cifar_runs = [
    dict(name="cosine_50", epochs=50, schedule="cosine", max_lr=None, lr=0.1),
    dict(name="onecycle_50", epochs=50, max_lr=0.2, lr=0.02),
    dict(name="onecycle_20", epochs=20, max_lr=0.2, lr=0.02),
]
cifar_results, cifar_history = [], []
for seed in seeds:
    for config in cifar_runs:
        history, scores = train(cifar, seed=seed, output="results/cifar", **config)
        cifar_history.append(history.assign(method=config["name"], seed=seed))
        cifar_results.append(scores)
cifar_results = pd.concat(cifar_results, ignore_index=True)
cifar_history = pd.concat(cifar_history, ignore_index=True)
cifar_results.groupby("method").agg(accuracy=("test_accuracy", "mean"), std=("test_accuracy", "std"), seconds=("train_seconds", "mean"))

In [ ]:
plot_curves(cifar_history, "epoch", ["train_eval_loss", "validation_loss", "validation_accuracy"])
plot_curves(read_diagnostics("results/cifar"), "step", ["weight_l2", "grad_l2", "update_ratio"])

## Проверка регуляризующего эффекта

На фиксированных 10 000 train-примерах MNIST проверим сетку из двух пиков LR и трех weight decay. Все режимы — 40 эпох 1cycle с постоянным momentum=0,9: меняются только пик LR и weight decay. Используем LeNet без BatchNorm, чтобы масштабная инвариантность нормализации не мешала трактовать нормы.

Смотрим, уменьшает ли высокий LR переобучение и меняется ли полезный уровень weight decay. Меньший gap при плохом train и validation может означать недообучение. Похожее качество или меньшая L2-норма не доказывают эквивалентность L2-регуляризации.

In [ ]:
reg_data = load_data(device, train_size=10000)
reg_results, reg_history = [], []
for seed in seeds:
    for peak in [0.02, 0.1]:
        for wd in [0.0, 0.0005, 0.005]:
            name = f"lr={peak}_wd={wd}"
            history, scores = train(reg_data, name=name, epochs=40, max_lr=peak, seed=seed,
                                    weight_decay=wd, cycle_momentum=False, output="results/regularization")
            reg_history.append(history.assign(method=name, seed=seed, peak=peak, weight_decay=wd))
            reg_results.append(scores.assign(peak=peak, weight_decay=wd))
reg_history = pd.concat(reg_history, ignore_index=True)
reg_results = pd.concat(reg_results, ignore_index=True)

In [ ]:
last_epoch = reg_history.loc[reg_history.epoch == 40]
display(last_epoch.groupby(["peak", "weight_decay"])[["train_eval_loss", "validation_loss", "gap", "validation_accuracy"]].mean())
plot_curves(reg_history, "epoch", ["train_eval_loss", "validation_loss", "gap"])
reg_norms = read_diagnostics("results/regularization")
plot_curves(reg_norms, "step", ["weight_l2", "grad_l2", "update_ratio"])
plot_curves(reg_norms, "step", ["rms/features.0.weight", "rms/classifier.1.weight"])

## Часть 2. DQN в LunarLander-v3

Stable-Baselines3, Adam с постоянными betas, настройки из RL Zoo. Меняем только расписание LR: constant=0,00063; cosine от этого значения; 1cycle с пиком 0,00189. Это перенос идеи расписания LR, а не всего SGD-рецепта статьи.

Каждый запуск — 100 000 шагов среды. Exploration, replay buffer и target network одинаковы. Кривые награды считаются на одних и тех же 10 eval-seed; финальные веса проверяются на других 50 seed. Лучший checkpoint по eval не выбираем. Три training seed — предварительное сравнение. Маленькая сеть и Box2D запускаются на CPU.

TD-loss имеет движущуюся цель, поэтому качество оцениваем по награде. В диагностике RL `grad_l2_clipped` — норма после clipping; `update_ratio` относится к блоку из 4 обновлений, а не к одному шагу SGD.

In [ ]:
from rl import train_rl

rl_results, rl_history = [], []
for seed in seeds:
    for kind in ["constant", "cosine", "onecycle"]:
        history, result = train_rl(kind, seed, steps=100000)
        rl_history.append(history.assign(method=kind, seed=seed))
        rl_results.append(result)
rl_results = pd.DataFrame(rl_results)
rl_history = pd.concat(rl_history, ignore_index=True)
rl_results.groupby("method").agg(reward=("reward_mean", "mean"), between_seed_std=("reward_mean", "std"), seconds=("train_seconds", "mean"))

In [ ]:
plot_curves(rl_history, "step", ["reward_mean"])
rl_norms = read_diagnostics("results/rl")
plot_curves(rl_norms, "step", ["td_loss", "weight_l2", "grad_l2_clipped"])
plot_curves(rl_norms, "step", ["lr", "update_ratio"])

## Что сравним после запусков

1. Ускоряет ли 1cycle обучение на обоих датасетах при близкой точности?
2. Как пик LR взаимодействует с weight decay и разрывом train–validation?
3. Совпадают ли изменения loss с ростом градиентов и размера обновлений?
4. Переносится ли эффект в DQN или большой LR ухудшает устойчивость?

Результаты могут не подтвердить гипотезу. Нормы весов не измеряют плоскость минимума; сравнивать их численно между LeNet, ResNet и DQN не нужно.

[Статья](https://arxiv.org/abs/1708.07120) · [SB3 DQN](https://stable-baselines3.readthedocs.io/en/v2.7.0/modules/dqn.html) · [RL Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml)